In [1]:
import pandas as pd
import numpy as np

hcahps = pd.read_csv(
    'HCAHPS-Hospital.csv',
    dtype={'Facility ID': str},
    low_memory=False
)

print(f"Raw shape: {hcahps.shape}")
hcahps.head()

Raw shape: (325652, 22)


,Facility ID,Facility Name,Address,City/Town,State,ZIP Code,County/Parish,Telephone Number,HCAHPS Measure ID,HCAHPS Question,...,Patient Survey Star Rating Footnote,HCAHPS Answer Percent,HCAHPS Answer Percent Footnote,HCAHPS Linear Mean Value,Number of Completed Surveys,Number of Completed Surveys Footnote,Survey Response Rate Percent,Survey Response Rate Percent Footnote,Start Date,End Date
0,010001,SOUTHEAST HEALTH MEDICAL CENTER,1108 ROSS CLARK CIRCLE,DOTHAN,AL,36301,HOUSTON,(334) 793-8701,H_COMP_1_A_P,"Patients who reported that their nurses ""Alway...",...,NaN,75,NaN,Not Applicable,596,NaN,16,NaN,04/01/2024,03/31/2025
1,010001,SOUTHEAST HEALTH MEDICAL CENTER,1108 ROSS CLARK CIRCLE,DOTHAN,AL,36301,HOUSTON,(334) 793-8701,H_COMP_1_SN_P,"Patients who reported that their nurses ""Somet...",...,NaN,6,NaN,Not Applicable,596,NaN,16,NaN,04/01/2024,03/31/2025
2,010001,SOUTHEAST HEALTH MEDICAL CENTER,1108 ROSS CLARK CIRCLE,DOTHAN,AL,36301,HOUSTON,(334) 793-8701,H_COMP_1_U_P,"Patients who reported that their nurses ""Usual...",...,NaN,19,NaN,Not Applicable,596,NaN,16,NaN,04/01/2024,03/31/2025
3,010001,SOUTHEAST HEALTH MEDICAL CENTER,1108 ROSS CLARK CIRCLE,DOTHAN,AL,36301,HOUSTON,(334) 793-8701,H_COMP_1_LINEAR_SCORE,Nurse communication - linear mean score,...,NaN,Not Applicable,NaN,90,596,NaN,16,NaN,04/01/2024,03/31/2025
4,010001,SOUTHEAST HEALTH MEDICAL CENTER,1108 ROSS CLARK CIRCLE,DOTHAN,AL,36301,HOUSTON,(334) 793-8701,H_COMP_1_STAR_RATING,Nurse communication - star rating,...,NaN,Not Applicable,NaN,Not Applicable,596,NaN,16,NaN,04/01/2024,03/31/2025


In [2]:
print("--- Raw nulls ---")
print(hcahps.isnull().sum())

print("\n--- Raw dtypes ---")
print(hcahps.dtypes)

--- Raw nulls ---
Facility ID                                   0
Facility Name                                 0
Address                                       0
City/Town                                     0
State                                         0
ZIP Code                                      0
County/Parish                                 0
Telephone Number                              0
HCAHPS Measure ID                             0
HCAHPS Question                               0
HCAHPS Answer Description                     0
Patient Survey Star Rating                    0
Patient Survey Star Rating Footnote      311198
HCAHPS Answer Percent                         0
HCAHPS Answer Percent Footnote           239564
HCAHPS Linear Mean Value                      0
Number of Completed Surveys                   0
Number of Completed Surveys Footnote     210868
Survey Response Rate Percent                  0
Survey Response Rate Percent Footnote    210868
Start Date            

In [3]:
hcahps.columns = (
    hcahps.columns
    .str.strip()
    .str.lower()
    .str.replace(' ', '_')
    .str.replace('/', '_')
    .str.replace(r'[^\w]', '_', regex=True)
)

print(hcahps.columns.tolist())

['facility_id', 'facility_name', 'address', 'city_town', 'state', 'zip_code', 'county_parish', 'telephone_number', 'hcahps_measure_id', 'hcahps_question', 'hcahps_answer_description', 'patient_survey_star_rating', 'patient_survey_star_rating_footnote', 'hcahps_answer_percent', 'hcahps_answer_percent_footnote', 'hcahps_linear_mean_value', 'number_of_completed_surveys', 'number_of_completed_surveys_footnote', 'survey_response_rate_percent', 'survey_response_rate_percent_footnote', 'start_date', 'end_date']


In [4]:
CMS_NULLS = {
    'Not Available': np.nan,
    'Not Applicable': np.nan,
    'Too Few to Report': np.nan,
    'N/A': np.nan
}

hcahps = hcahps.replace(CMS_NULLS)

print("CMS placeholders replaced ")
print(f"Nulls after replace:\n{hcahps.isnull().sum()}")

CMS placeholders replaced 
Nulls after replace:
facility_id                                   0
facility_name                                 0
address                                       0
city_town                                     0
state                                         0
zip_code                                      0
county_parish                                 0
telephone_number                              0
hcahps_measure_id                             0
hcahps_question                               0
hcahps_answer_description                     0
patient_survey_star_rating               297005
patient_survey_star_rating_footnote      311198
hcahps_answer_percent                    132685
hcahps_answer_percent_footnote           239564
hcahps_linear_mean_value                 300188
number_of_completed_surveys               56304
number_of_completed_surveys_footnote     210868
survey_response_rate_percent              56304
survey_response_rate_percent_footnote   

In [5]:
hcahps['facility_id'] = (
    hcahps['facility_id']
    .str.strip()
    .str.zfill(6)
)

print("facility_id sample:", hcahps['facility_id'].head().tolist())

facility_id sample: ['010001', '010001', '010001', '010001', '010001']


In [6]:
numeric_cols = [
    'patient_survey_star_rating',
    'hcahps_answer_percent',
    'hcahps_linear_mean_value',
    'number_of_completed_surveys',
    'survey_response_rate_percent'
]

for col in numeric_cols:
    hcahps[col] = pd.to_numeric(hcahps[col], errors='coerce')

print("Numeric dtypes after cast:")
print(hcahps[numeric_cols].dtypes)

Numeric dtypes after cast:
patient_survey_star_rating      float64
hcahps_answer_percent           float64
hcahps_linear_mean_value        float64
number_of_completed_surveys     float64
survey_response_rate_percent    float64
dtype: object


In [7]:
footnote_cols = [
    'patient_survey_star_rating_footnote',
    'hcahps_answer_percent_footnote',
    'number_of_completed_surveys_footnote',
    'survey_response_rate_percent_footnote'
]

hcahps['is_suppressed'] = hcahps[footnote_cols].notna().any(axis=1)

print("Suppression breakdown:")
print(hcahps['is_suppressed'].value_counts())

Suppression breakdown:
is_suppressed
False    210868
True     114784
Name: count, dtype: int64


In [8]:
cols_to_drop = [
    'facility_name', 'address', 'city_town',
    'state', 'zip_code', 'county_parish', 'telephone_number',
    'start_date', 'end_date',
    'hcahps_question',          
    'hcahps_answer_description', 
    'patient_survey_star_rating_footnote',
    'hcahps_answer_percent_footnote',
    'number_of_completed_surveys_footnote',
    'survey_response_rate_percent_footnote'
]

hcahps = hcahps.drop(columns=cols_to_drop, errors='ignore')

print(f"Shape after dropping: {hcahps.shape}")
print(hcahps.columns.tolist())

Shape after dropping: (325652, 8)
['facility_id', 'hcahps_measure_id', 'patient_survey_star_rating', 'hcahps_answer_percent', 'hcahps_linear_mean_value', 'number_of_completed_surveys', 'survey_response_rate_percent', 'is_suppressed']


In [9]:
dupes = hcahps.duplicated(subset=['facility_id', 'hcahps_measure_id']).sum()
print(f"True duplicates on composite key: {dupes}")

True duplicates on composite key: 0


In [10]:
print(f"Shape           : {hcahps.shape}")
print(f"Unique hospitals: {hcahps['facility_id'].nunique()}")
print(f"Unique measures : {hcahps['hcahps_measure_id'].nunique()}")

print("\n--- Nulls ---")
print(hcahps.isnull().sum())

print("\n--- Measure IDs available ---")
print(hcahps['hcahps_measure_id'].value_counts())

Shape           : (325652, 8)
Unique hospitals: 4789
Unique measures : 68

--- Nulls ---
facility_id                          0
hcahps_measure_id                    0
patient_survey_star_rating      297005
hcahps_answer_percent           132685
hcahps_linear_mean_value        300188
number_of_completed_surveys      56304
survey_response_rate_percent     56304
is_suppressed                        0
dtype: int64

--- Measure IDs available ---
hcahps_measure_id
H_COMP_1_A_P             4789
H_DISCH_HELP_N_P         4789
H_CLEAN_HSP_U_P          4789
H_CLEAN_HSP_SN_P         4789
H_CLEAN_HSP_A_P          4789
                         ... 
H_COMP_5_A_P             4789
H_COMP_5_SN_P            4789
H_COMP_5_U_P             4789
H_COMP_5_LINEAR_SCORE    4789
H_STAR_RATING            4789
Name: count, Length: 68, dtype: int64


In [11]:

hcahps_pct = hcahps.pivot_table(
    index='facility_id',
    columns='hcahps_measure_id',
    values='hcahps_answer_percent',
    aggfunc='first'
).reset_index()

hcahps_pct.columns = (
    ['facility_id'] +
    ['hcahps_pct_' + col.lower() for col in hcahps_pct.columns[1:]]
)

hcahps_linear = hcahps.pivot_table(
    index='facility_id',
    columns='hcahps_measure_id',
    values='hcahps_linear_mean_value',
    aggfunc='first'
).reset_index()

hcahps_linear.columns = (
    ['facility_id'] +
    ['hcahps_linear_' + col.lower() for col in hcahps_linear.columns[1:]]
)

hcahps_stars = hcahps.pivot_table(
    index='facility_id',
    columns='hcahps_measure_id',
    values='patient_survey_star_rating',
    aggfunc='first'
).reset_index()

hcahps_stars.columns = (
    ['facility_id'] +
    ['hcahps_stars_' + col.lower() for col in hcahps_stars.columns[1:]]
)

hcahps_wide = hcahps_pct.merge(hcahps_linear, on='facility_id', how='left')
hcahps_wide = hcahps_wide.merge(hcahps_stars, on='facility_id', how='left')

print(f"Wide table shape: {hcahps_wide.shape}")
hcahps_wide.head()

Wide table shape: (3961, 69)


,facility_id,hcahps_pct_h_clean_hsp_a_p,hcahps_pct_h_clean_hsp_sn_p,hcahps_pct_h_clean_hsp_u_p,hcahps_pct_h_comp_1_a_p,hcahps_pct_h_comp_1_sn_p,hcahps_pct_h_comp_1_u_p,hcahps_pct_h_comp_2_a_p,hcahps_pct_h_comp_2_sn_p,hcahps_pct_h_comp_2_u_p,...,hcahps_linear_h_recmnd_linear_score,hcahps_stars_h_clean_star_rating,hcahps_stars_h_comp_1_star_rating,hcahps_stars_h_comp_2_star_rating,hcahps_stars_h_comp_5_star_rating,hcahps_stars_h_comp_6_star_rating,hcahps_stars_h_hsp_rating_star_rating,hcahps_stars_h_quiet_star_rating,hcahps_stars_h_recmnd_star_rating,hcahps_stars_h_star_rating
0,010001,72.0,11.0,17.0,75.0,6.0,19.0,82.0,4.0,14.0,...,91.0,3.0,3.0,4.0,3.0,4.0,4.0,4.0,5.0,4.0
1,010005,68.0,12.0,20.0,78.0,5.0,17.0,82.0,4.0,14.0,...,85.0,3.0,3.0,4.0,2.0,3.0,3.0,4.0,3.0,3.0
2,010006,61.0,17.0,22.0,75.0,6.0,19.0,73.0,8.0,19.0,...,80.0,2.0,2.0,2.0,1.0,2.0,2.0,4.0,2.0,2.0
3,010007,62.0,16.0,22.0,77.0,6.0,17.0,84.0,3.0,13.0,...,84.0,2.0,3.0,4.0,3.0,4.0,3.0,5.0,3.0,3.0
4,010008,76.0,8.0,16.0,89.0,3.0,8.0,86.0,5.0,9.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [12]:
hcahps_wide.to_csv('hcahps_cleaned.csv', index=False)
print("Saved as hcahps_cleaned.csv")

Saved as hcahps_cleaned.csv


In [13]:
print(hcahps_wide.columns.tolist())

['facility_id', 'hcahps_pct_h_clean_hsp_a_p', 'hcahps_pct_h_clean_hsp_sn_p', 'hcahps_pct_h_clean_hsp_u_p', 'hcahps_pct_h_comp_1_a_p', 'hcahps_pct_h_comp_1_sn_p', 'hcahps_pct_h_comp_1_u_p', 'hcahps_pct_h_comp_2_a_p', 'hcahps_pct_h_comp_2_sn_p', 'hcahps_pct_h_comp_2_u_p', 'hcahps_pct_h_comp_5_a_p', 'hcahps_pct_h_comp_5_sn_p', 'hcahps_pct_h_comp_5_u_p', 'hcahps_pct_h_comp_6_n_p', 'hcahps_pct_h_comp_6_y_p', 'hcahps_pct_h_disch_help_n_p', 'hcahps_pct_h_disch_help_y_p', 'hcahps_pct_h_doctor_explain_a_p', 'hcahps_pct_h_doctor_explain_sn_p', 'hcahps_pct_h_doctor_explain_u_p', 'hcahps_pct_h_doctor_listen_a_p', 'hcahps_pct_h_doctor_listen_sn_p', 'hcahps_pct_h_doctor_listen_u_p', 'hcahps_pct_h_doctor_respect_a_p', 'hcahps_pct_h_doctor_respect_sn_p', 'hcahps_pct_h_doctor_respect_u_p', 'hcahps_pct_h_hsp_rating_0_6', 'hcahps_pct_h_hsp_rating_7_8', 'hcahps_pct_h_hsp_rating_9_10', 'hcahps_pct_h_med_for_a_p', 'hcahps_pct_h_med_for_sn_p', 'hcahps_pct_h_med_for_u_p', 'hcahps_pct_h_nurse_explain_a_p', 'hc